# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# The metadata attribute is an object with .to_json()
metadata = dataset.metadata.to_json()

print(metadata['name'])
print('-' * 80)
print(metadata['description'])

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

We'll enumerate:
- All record sets (`@id`)
- For one example record set, its fields and field `@id`s
- For one field, the columns contained

This helps you explore what's available and how to reference specific data entities by `@id`.

In [ ]:
# List all record sets with their @id and name
print('Record sets:')
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"  - @id: {rs.id}, name: {rs.name}")

# Choose the first record set to inspect fields
if len(record_sets) > 0:
    example_rs = record_sets[0]
    print(f"\nFields in record set '{example_rs.name}' (ID: {example_rs.id}):")
    for field in example_rs.fields:
        print(f"  - @id: {field.id}, name: {field.name}, data type: {field.data_type}")

    # For one field, show the columns
    if len(example_rs.fields) > 0:
        field = example_rs.fields[0]
        print(f"\nColumns of field '{field.name}' (@id: {field.id}):")
        for column in field.columns:
            print(f"  - @id: {column.id}, name: {column.name}, source: {column.source}")

## 3. Data Extraction
Load data from one or more record sets into a DataFrame for analysis. Refer to record set and field `@id`s as listed above.

We'll demonstrate extraction for all available record sets.

In [ ]:
# Extract all data from each record set
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Each record set's records
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}.")

# Show the first record set's dataframe info
if len(record_set_ids) > 0 and record_set_ids[0] in dataframes:
    first_rs_id = record_set_ids[0]
    print(f"\nColumns in record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's demonstrate a few common EDA tasks:
- Filter records by a numeric field (e.g., `score_gad7`)
- Normalize the numeric field
- Group by a demographic field (e.g., `gender` if available)

**Note:** You may need to adjust field names based on the available field `@id`s listed earlier.

In [ ]:
# Example: Use the main survey record set and fields 'score_gad7' and 'gender'

# Replace with your actual record set and field IDs if they differ
record_set_id = None
numeric_field_id = None
group_field_id = None

# Attempt to infer common field names/IDs
for rs in dataset.metadata.record_sets:
    if any('gad' in (f.id.lower() if f.id else '') for f in rs.fields):
        record_set_id = rs.id
        for f in rs.fields:
            if 'gad' in (f.id.lower() if f.id else ''):
                numeric_field_id = f.id
            if 'gender' in (f.id.lower() if f.id else ''):
                group_field_id = f.id
        break

if (record_set_id and numeric_field_id):
    df = dataframes[record_set_id]
    print(f"Operating on record_set = {record_set_id}, numeric_field = {numeric_field_id}, group_field = {group_field_id}")

    # Drop NaNs for clean filtering
    df = df.copy()
    try:
        # Try to ensure numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = 5  # example threshold for GAD-7 scores
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
        display(filtered_df.head())

        # Normalize
        normcol = f"{numeric_field_id}_normalized"
        filtered_df[normcol] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, normcol]].head())

        # Group by its group_field if possible
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id, dropna=True)[numeric_field_id].mean().to_frame('mean_score')
            print(f"Mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df)
    except Exception as e:
        print("An error occurred in EDA:", e)
else:
    print('Could not auto-detect a record set with a GAD-7 numeric field or gender field. Please refer to Section 2 to find valid field @ids and update them here.')

## 5. Visualization
Visualize data distributions or relationships between fields in this dataset.

For example, let's visualize the distribution of GAD-7 scores, and, if available, show means by gender.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if (record_set_id and numeric_field_id and record_set_id in dataframes):
    df = dataframes[record_set_id]
    try:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=10)
        plt.xlabel(numeric_field_id)
        plt.title(f"Distribution of {numeric_field_id} in {record_set_id}")
        plt.show()

        if group_field_id and group_field_id in df.columns:
            plt.figure(figsize=(8,4))
            sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.show()
    except Exception as e:
        print("Could not plot visualization due to error: ", e)
else:
    print('No suitable field IDs detected for visualization. See previous sections to select appropriate field @ids.')

## 6. Conclusion
In this notebook, we have:
- Loaded rich metadata and tabular data from a FAIR² Mental Health Survey Croissant dataset
- Explored record sets, fields, and columns using entity `@id`
- Performed extraction and simple EDA referencing the Croissant schema for field IDs
- Visualized key variables (e.g., GAD-7 score distributions)

You can further extend this notebook by exploring more fields, record sets, or building custom analyses guided by the Croissant schema.